# PROCESO DE LIMPIEZA COMPLETO
## Persona 2: Ingeniero de Calidad y Limpieza de Datos

---

### Objetivo:
Limpiar el dataset completo de 10,000 registros eliminando los 5 tipos de errores:
1. Valores faltantes → Imputar con mediana
2. Typos → Corregir con strip()
3. Duplicados → Eliminar
4. Fechas inconsistentes → Eliminar
5. Outliers extremos → Eliminar (IQR x3)

### Resultado esperado:
- Dataset limpio: ~9,500 registros (95% retencion)
- 0 valores faltantes
- 0 duplicados
- 100% fechas validas
- Sin outliers extremos

---
## 1. CONFIGURACION E IMPORTACION DEL MODULO

In [1]:
import pandas as pd
import numpy as np
import sys

# Agregar la ruta del modulo src
sys.path.append('../src')

# Importar el modulo de limpieza
import data_cleaning as dc

print("Modulo de limpieza importado correctamente")

Modulo de limpieza importado correctamente


---
## 2. CARGAR DATASET COMPLETO (10,000 REGISTROS)

In [2]:
# Cargar el dataset completo de 10,000 registros
df_raw = pd.read_csv('../data/raw/dataset_raw.csv')

print(f"Dataset cargado: {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas")
print(f"\nPrimeras 3 filas:")
df_raw.head(3)

Dataset cargado: 10000 filas x 39 columnas

Primeras 3 filas:


,order_id,customer_id,order_date,shipped_date,delivered_date,processing_days,delivery_days,promised_delivery_days,actual_delivery_days,delivery_delay_days,...,payment_installments,sales_channel,platform_name,is_peak_season,customer_delivery_rating,delivery_issue,shipping_cost_to_price_ratio,is_frequent_customer,is_loyal_customer,high_value_order
0,ECOMMX-2024-00000,CUST-25604,2025-03-21 14:47:03,2025-03-23 14:47:03,2025-03-28 14:47:03,2,5,6,5,-1,...,1,Marketplace,Liverpool,0,5,Ninguno,0.1059,0,1,1
1,ECOMMX-2024-00001,CUST-93042,2025-12-22 02:25:19,2025-12-23 02:25:19,2025-12-25 02:25:19,1,2,2,2,0,...,1,Marketplace,Liverpool,1,5,Ninguno,0.0284,1,0,0
2,ECOMMX-2024-00002,CUST-64249,2025-09-05 03:26:12,2025-09-08 03:26:12,2025-09-10 03:26:12,3,2,4,2,-2,...,1,Marketplace,Amazon México,0,5,Ninguno,0.8629,1,0,0


---
## 3. ANALISIS DETALLADO DE ERRORES

Antes de limpiar, analicemos cada tipo de error en detalle.

### 3.1 Valores Faltantes

In [3]:
print("ANALISIS DE VALORES FALTANTES")
print("="*50)

missing_info = dc.detectar_valores_faltantes(df_raw)
print(missing_info)

print(f"\nTotal de valores faltantes: {df_raw.isnull().sum().sum()}")
print(f"Columnas afectadas: {len(missing_info)}")

ANALISIS DE VALORES FALTANTES
                    Columna  Valores_Faltantes  Porcentaje
18        shipping_cost_mxn                 28        0.28
19              distance_km                 27        0.27
12        product_price_mxn                 21        0.21
24  customer_loyalty_months                 21        0.21

Total de valores faltantes: 97
Columnas afectadas: 4


### 3.2 Typos en Transportistas

In [4]:
print("ANALISIS DE TYPOS")
print("="*50)

print("\nDistribucion de transportistas (incluyendo typos):")
print(df_raw['shipping_carrier'].value_counts())

typos = dc.detectar_typos_transportistas(df_raw)
print(f"\nTransportistas con typos detectados: {len(typos)}")
for typo in typos:
    count = (df_raw['shipping_carrier'] == typo).sum()
    print(f"  '{typo}' -> {count} registros")

ANALISIS DE TYPOS

Distribucion de transportistas (incluyendo typos):
shipping_carrier
Estafeta              2781
DHL                   2229
FedEx                 1817
Correos de México     1448
UPS                    991
Redpack                480
Paquetexpress          197
 Correos de México      14
 FedEx                  12
 Estafeta               12
 DHL                    12
 UPS                     3
 Redpack                 2
 Paquetexpress           2
Name: count, dtype: int64

Transportistas con typos detectados: 7
  ' FedEx' -> 12 registros
  ' Estafeta' -> 12 registros
  ' Correos de México' -> 14 registros
  ' DHL' -> 12 registros
  ' Redpack' -> 2 registros
  ' Paquetexpress' -> 2 registros
  ' UPS' -> 3 registros


### 3.3 Duplicados

In [5]:
print("ANALISIS DE DUPLICADOS")
print("="*50)

duplicados = dc.detectar_duplicados(df_raw)
print(f"Registros duplicados encontrados: {len(duplicados)}")

if len(duplicados) > 0:
    print("\nEjemplos de registros duplicados:")
    print(duplicados[['order_id', 'customer_id', 'order_date']].head())

ANALISIS DE DUPLICADOS
Registros duplicados encontrados: 0


### 3.4 Fechas Inconsistentes

In [6]:
print("ANALISIS DE FECHAS INCONSISTENTES")
print("="*50)

fechas_inv = dc.detectar_fechas_inconsistentes(df_raw)
print(f"Registros con fechas inconsistentes: {len(fechas_inv)}")

if len(fechas_inv) > 0:
    print("\nEjemplos de fechas inconsistentes:")
    print(fechas_inv[['order_id', 'order_date', 'shipped_date', 'delivered_date']].head())
    
    # Analizar tipos de inconsistencias
    envio_antes_pedido = (fechas_inv['shipped_date'] < fechas_inv['order_date']).sum()
    entrega_antes_envio = (fechas_inv['delivered_date'] < fechas_inv['shipped_date']).sum()
    
    print(f"\nTipos de inconsistencias:")
    print(f"  - Envio antes del pedido: {envio_antes_pedido}")
    print(f"  - Entrega antes del envio: {entrega_antes_envio}")

ANALISIS DE FECHAS INCONSISTENTES
Registros con fechas inconsistentes: 104

Ejemplos de fechas inconsistentes:
              order_id          order_date        shipped_date  \
193  ECOMMX-2024-00193 2025-04-19 18:22:39 2025-04-21 18:22:39   
269  ECOMMX-2024-00269 2025-03-19 07:49:16 2025-03-20 07:49:16   
311  ECOMMX-2024-00311 2025-10-02 18:55:31 2025-10-01 18:55:31   
462  ECOMMX-2024-00462 2025-09-15 03:29:17 2025-09-14 03:29:17   
592  ECOMMX-2024-00592 2025-11-24 10:01:22 2025-11-25 10:01:22   

         delivered_date  
193 2025-04-19 18:22:39  
269 2025-03-18 07:49:16  
311 2025-10-07 18:55:31  
462 2025-09-19 03:29:17  
592 2025-11-23 10:01:22  

Tipos de inconsistencias:
  - Envio antes del pedido: 40
  - Entrega antes del envio: 64


### 3.5 Outliers en Precios

In [7]:
print("ANALISIS DE OUTLIERS EN PRECIOS")
print("="*50)

print("\nEstadisticas de precios:")
print(df_raw['product_price_mxn'].describe())

outliers_precio = dc.detectar_outliers_precio(df_raw)
print(f"\nOutliers detectados: {len(outliers_precio)}")

if len(outliers_precio) > 0:
    print("\nEjemplos de outliers extremos:")
    print(outliers_precio[['order_id', 'product_category', 'product_price_mxn']].nlargest(10, 'product_price_mxn'))

ANALISIS DE OUTLIERS EN PRECIOS

Estadisticas de precios:
count    9.979000e+03
mean     9.188263e+03
std      5.872904e+04
min      5.103000e+01
25%      1.550630e+03
50%      3.279770e+03
75%      8.835045e+03
max      3.291971e+06
Name: product_price_mxn, dtype: float64

Outliers detectados: 1086

Ejemplos de outliers extremos:
               order_id product_category  product_price_mxn
4073  ECOMMX-2024-04073     Electrónicos          3291971.0
2244  ECOMMX-2024-02244     Electrónicos          3008606.0
7290  ECOMMX-2024-07290     Electrónicos          1548066.0
3979  ECOMMX-2024-03979   Hogar y Jardín          1397853.0
3928  ECOMMX-2024-03928   Hogar y Jardín          1245014.0
8731  ECOMMX-2024-08731   Hogar y Jardín          1173112.0
6536  ECOMMX-2024-06536     Electrónicos          1135800.0
2985  ECOMMX-2024-02985     Electrónicos          1094735.0
7210  ECOMMX-2024-07210     Electrónicos          1034436.0
3856  ECOMMX-2024-03856   Hogar y Jardín           932394.0


### 3.6 Outliers en Distancias

In [8]:
print("ANALISIS DE OUTLIERS EN DISTANCIAS")
print("="*50)

print("\nEstadisticas de distancias:")
print(df_raw['distance_km'].describe())

outliers_dist = dc.detectar_outliers_distancia(df_raw)
print(f"\nOutliers detectados: {len(outliers_dist)}")

if len(outliers_dist) > 0:
    print("\nEjemplos de distancias imposibles:")
    print(outliers_dist[['order_id', 'customer_state', 'distance_km']].nlargest(10, 'distance_km'))

ANALISIS DE OUTLIERS EN DISTANCIAS

Estadisticas de distancias:
count    9973.000000
mean      699.545573
std       651.945660
min         0.000000
25%       250.000000
50%       630.000000
75%      1064.000000
max      9791.000000
Name: distance_km, dtype: float64

Outliers detectados: 46

Ejemplos de distancias imposibles:
               order_id  customer_state  distance_km
5389  ECOMMX-2024-05389        Coahuila       9791.0
7765  ECOMMX-2024-07765         Morelos       9775.0
4341  ECOMMX-2024-04341        Campeche       9736.0
6483  ECOMMX-2024-06483      Tamaulipas       9718.0
6241  ECOMMX-2024-06241         Morelos       9581.0
2742  ECOMMX-2024-02742        Veracruz       9554.0
9074  ECOMMX-2024-09074          Oaxaca       9513.0
5547  ECOMMX-2024-05547  Aguascalientes       9480.0
9106  ECOMMX-2024-09106         Tabasco       9380.0
2173  ECOMMX-2024-02173        Coahuila       9366.0


### 3.7 Resumen de Errores Antes de Limpiar

In [9]:
print("RESUMEN DE ERRORES DETECTADOS")
print("="*70)

print(f"\nTotal de registros: {len(df_raw)}")
print(f"\n1. Valores faltantes: {df_raw.isnull().sum().sum()} valores")
print(f"2. Typos en transportistas: {len(typos)} tipos detectados")
print(f"3. Duplicados: {len(duplicados)} registros")
print(f"4. Fechas inconsistentes: {len(fechas_inv)} registros")
print(f"5. Outliers en precios: {len(outliers_precio)} registros")
print(f"6. Outliers en distancias: {len(outliers_dist)} registros")

# Estimar registros a eliminar
registros_a_eliminar = len(fechas_inv) + len(outliers_precio.merge(outliers_dist, on='order_id', how='outer'))
print(f"\nRegistros estimados a eliminar: ~{registros_a_eliminar}")
print(f"Retencion estimada: ~{((len(df_raw) - registros_a_eliminar) / len(df_raw)) * 100:.1f}%")

RESUMEN DE ERRORES DETECTADOS

Total de registros: 10000

1. Valores faltantes: 97 valores
2. Typos en transportistas: 7 tipos detectados
3. Duplicados: 0 registros
4. Fechas inconsistentes: 104 registros
5. Outliers en precios: 1086 registros
6. Outliers en distancias: 46 registros

Registros estimados a eliminar: ~1227
Retencion estimada: ~87.7%


---
## 4. APLICAR LIMPIEZA COMPLETA

La funcion `limpiar_dataset_completo()` aplica todas las limpiezas en este orden:
1. Corregir typos (no elimina registros)
2. Imputar valores faltantes (no elimina registros)
3. Eliminar duplicados
4. Eliminar fechas inconsistentes
5. Eliminar outliers extremos (IQR x3)

In [10]:
# Aplicar todas las funciones de limpieza
df_clean = dc.limpiar_dataset_completo(df_raw)

Iniciando proceso de limpieza...
Registros iniciales: 10000
1. Typos corregidos
2. Valores faltantes imputados
3. Duplicados eliminados: 0
4. Fechas inconsistentes eliminadas: 104
5. Outliers eliminados: 381

Registros finales: 9515
Registros eliminados totales: 485


---
## 5. GENERAR REPORTE DE LIMPIEZA

In [11]:
# Generar reporte comparativo
reporte = dc.generar_reporte_limpieza(df_raw, df_clean)

print("REPORTE DE LIMPIEZA")
print("="*50)
for key, value in reporte.items():
    if 'porcentaje' in key:
        print(f"{key}: {value:.2f}%")
    else:
        print(f"{key}: {value}")

REPORTE DE LIMPIEZA
registros_originales: 10000
registros_limpios: 9515
registros_eliminados: 485
porcentaje_retenido: 95.15%
valores_faltantes_antes: 97
valores_faltantes_despues: 0


---
## 6. VERIFICACION DE CALIDAD FINAL

In [12]:
# Verificar que no queden problemas
print("VERIFICACION FINAL")
print("="*50)
print(f"\nValores faltantes: {df_clean.isnull().sum().sum()}")
print(f"Duplicados en order_id: {df_clean['order_id'].duplicated().sum()}")
print(f"Transportistas unicos: {df_clean['shipping_carrier'].nunique()}")
print("\nTransportistas (sin typos):")
print(df_clean['shipping_carrier'].value_counts())

VERIFICACION FINAL

Valores faltantes: 0
Duplicados en order_id: 0
Transportistas unicos: 7

Transportistas (sin typos):
shipping_carrier
Estafeta             2669
DHL                  2124
FedEx                1746
Correos de México    1391
UPS                   939
Redpack               458
Paquetexpress         188
Name: count, dtype: int64


In [13]:
# Verificar fechas
print("\nVERIFICACION DE FECHAS")
print("="*50)

df_clean['order_date'] = pd.to_datetime(df_clean['order_date'])
df_clean['shipped_date'] = pd.to_datetime(df_clean['shipped_date'])
df_clean['delivered_date'] = pd.to_datetime(df_clean['delivered_date'])

fechas_inv_despues = df_clean[
    (df_clean['shipped_date'] < df_clean['order_date']) |
    (df_clean['delivered_date'] < df_clean['shipped_date'])
]

print(f"Fechas inconsistentes despues de limpieza: {len(fechas_inv_despues)}")
print("Todas las fechas son logicamente consistentes: ", len(fechas_inv_despues) == 0)


VERIFICACION DE FECHAS
Fechas inconsistentes despues de limpieza: 0
Todas las fechas son logicamente consistentes:  True


In [14]:
# Verificar outliers
print("\nVERIFICACION DE OUTLIERS")
print("="*50)

print("\nEstadisticas de precios DESPUES de limpieza:")
print(df_clean['product_price_mxn'].describe())

print("\nEstadisticas de distancias DESPUES de limpieza:")
print(df_clean['distance_km'].describe())

# Verificar que no hay valores extremos
precio_max = df_clean['product_price_mxn'].max()
distancia_max = df_clean['distance_km'].max()

print(f"\nPrecio maximo en dataset limpio: ${precio_max:,.2f} MXN")
print(f"Distancia maxima en dataset limpio: {distancia_max:,.0f} km")
print(f"\nDistancia maxima es razonable para Mexico: {distancia_max <= 3000}")


VERIFICACION DE OUTLIERS

Estadisticas de precios DESPUES de limpieza:
count     9515.000000
mean      6069.353159
std       7078.482252
min         51.030000
25%       1505.010000
50%       3118.400000
75%       7648.880000
max      30786.810000
Name: product_price_mxn, dtype: float64

Estadisticas de distancias DESPUES de limpieza:
count    9515.000000
mean      666.405991
std       446.246508
min         0.000000
25%       248.000000
50%       626.000000
75%      1056.500000
max      1499.000000
Name: distance_km, dtype: float64

Precio maximo en dataset limpio: $30,786.81 MXN
Distancia maxima en dataset limpio: 1,499 km

Distancia maxima es razonable para Mexico: True


---
## 7. GUARDAR DATASET LIMPIO

In [15]:
# Crear carpeta processed si no existe
import os
os.makedirs('../data/processed', exist_ok=True)

# Guardar dataset limpio
df_clean.to_csv('../data/processed/dataset_clean.csv', index=False)

print(f"Dataset limpio guardado en: data/processed/dataset_clean.csv")
print(f"Total de registros: {len(df_clean)}")
print(f"\nDataset listo para Persona 3 (Analisis Estadistico) y Persona 4 (Visualizacion)")

Dataset limpio guardado en: data/processed/dataset_clean.csv
Total de registros: 9515

Dataset listo para Persona 3 (Analisis Estadistico) y Persona 4 (Visualizacion)


---
## CONCLUSION

### Proceso completado exitosamente:

**Dataset Original:**
- 10,000 registros
- 97 valores faltantes
- 8 typos en transportistas
- ~104 fechas inconsistentes
- ~381 outliers extremos

**Dataset Limpio:**
- 9,515 registros (95.15% retencion)
- 0 valores faltantes
- 0 typos
- 0 fechas inconsistentes
- 0 outliers extremos
- 7 transportistas unicos correctos

**Decisiones tecnicas clave:**
- IQR x3 (conservador) en lugar de x1.5 (estandar)
- Imputacion con mediana para valores faltantes
- Eliminacion de registros con errores irrecuperables

**Archivos generados:**
- `data/processed/dataset_clean.csv` - Dataset listo para analisis
- `src/data_cleaning.py` - Modulo reutilizable
- `docs/limpieza_datos.md` - Documentacion completa